In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import requests
from pysus import sim


In [2]:
codigos_x = [f"X{i}" for i in range(85, 100)]
codigo_y = [f"Y0{i}" for i in range(0, 10)]
codigo_agressao = codigos_x + codigo_y


In [ ]:
estados = ["AC", "AL", "AM", "AP", "BA", "CE", "DF", "ES", "GO", 
        "MA", "MG", "MS", "MT", "PA", "PB", "PE", "PI", "PR", 
        "RJ", "RN", "RO", "RR", "RS", "SC", "SE", "SP", "TO"]

anos = list(range(2015, 2022))

if os.path.exists("violencia_feminina.parquet"):    
    violencia_feminina = pd.read_parquet("violencia_feminina.parquet")
    print("Dados carregados do arquivo parquet.")
else:
    dfs = []
    
    for estado in estados:
        for ano in anos:
            df_temp = sim(state=estado, year=ano)
            df_filtrado = df_temp[
                (df_temp["SEXO"] == "2") &
                (df_temp["CAUSABAS"].str[:3].isin(codigo_agressao))
            ]
            dfs.append(df_filtrado)
            del df_temp
                
    violencia_feminina = pd.concat(dfs, ignore_index=True)
    violencia_feminina.to_parquet("violencia_feminina.parquet")

violencia_feminina["ANO"] = pd.to_datetime(
violencia_feminina["DTOBITO"], format="%d%m%Y", errors="coerce"
    ).dt.year
violencia_feminina = violencia_feminina.dropna(subset=["ANO"])
violencia_feminina["ANO"] = violencia_feminina["ANO"].astype(int)

print(f"Total de registros: {violencia_feminina.shape[0]}")
print(f"total de colunas: {violencia_feminina.shape[1]}")


Dados carregados do arquivo parquet.
Total de registros: 33873
total de colunas: 89


In [4]:
violencia_feminina["ESTADO"] = violencia_feminina["CODMUNOCOR"].str.strip().str[:2]

codigo_estado = {
    "11": "RO", "12": "AC", "13": "AM", "14": "RR", "15": "PA",
    "16": "AP", "17": "TO", "21": "MA", "22": "PI", "23": "CE",
    "24": "RN", "25": "PB", "26": "PE", "27": "AL", "28": "SE",
    "29": "BA", "31": "MG", "32": "ES", "33": "RJ", "35": "SP",
    "41": "PR", "42": "SC", "43": "RS", "50": "MS", "51": "MT",
    "52": "GO", "53": "DF"
}

violencia_feminina["ESTADO"] = violencia_feminina["ESTADO"].map(codigo_estado)


In [7]:
url = "https://servicodados.ibge.gov.br/api/v3/agregados/6579/periodos/2015|2016|2017|2018|2019|2020|2021|2022/variaveis/9324?localidades=N3[all]"

response = requests.get(url)
dados = response.json()
print(response.status_code)

200


In [22]:
registros = []

for serie in dados[0]["resultados"][0]["series"]:
    estado = codigo_estado[serie["localidade"]["id"]]
    for ano, populacao in serie["serie"].items():
        registros.append({
            "ESTADO": estado,
            "ANO": int(ano),
            "POPULACAO": int(populacao)
        })
        

populacao_df = pd.DataFrame(registros)
print(populacao_df.head(5))
print(populacao_df.shape)


  ESTADO   ANO  POPULACAO
0     RO  2015    1768204
1     RO  2016    1787279
2     RO  2017    1805788
3     RO  2018    1757589
4     RO  2019    1777225
(189, 3)


In [9]:
url = "https://servicodados.ibge.gov.br/api/v3/agregados/6579/periodos/2015|2016|2017|2018|2019|2020|2021|2022/variaveis/9324?localidades=N3[all]"

response = requests.get(url)
dados = response.json()
